<a href="https://colab.research.google.com/github/shameke/Yernar_colab_230103189/blob/main/Distributed_Multi_Rank_Stencil_230103189.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!nvidia-smi
!mpirun --version | head -n 2
!pip install -q mpi4py numba numpy

Fri Nov 20 05:12:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
GITHUB_USERNAME = "shameke"

In [8]:
GITHUB_USERNAME = "shameke"

assert GITHUB_USERNAME != "REPLACE_WITH_YOUR_GITHUB_USERNAME"
assert len(GITHUB_USERNAME.strip()) > 0

print(f"Configured student GitHub identity: {GITHUB_USERNAME}")

Configured student GitHub identity: shameke


In [9]:
%%writefile distributed_cuda_solver.py

import hashlib
import subprocess
import numpy as np

from mpi4py import MPI
from numba import cuda


def get_gpu_uuid():
    """Get NVIDIA GPU UUID."""
    result = subprocess.check_output(
        [
            "nvidia-smi",
            "--query-gpu=uuid",
            "--format=csv,noheader"
        ],
        encoding="utf-8"
    )

    return result.strip().splitlines()[0]


def derive_student_seed(github_user, gpu_uuid):
    """Generate unique 64-bit seed."""
    digest = hashlib.sha256(
        f"{github_user.strip()}::{gpu_uuid}".encode("utf-8")
    ).hexdigest()

    return int(digest[:16], 16)


# =====================================================
# TASK 1: Distributed Cartesian Topology
# =====================================================

def init_distributed_context(github_user):

    # Initialize MPI
    comm = MPI.COMM_WORLD

    rank = comm.Get_rank()
    size = comm.Get_size()

    # Check exactly 4 MPI processes
    if size != 4:
        raise RuntimeError(
            f"Expected exactly 4 MPI ranks, got {size}"
        )

    # Create 2x2 Cartesian topology
    dims = [2, 2]
    periods = [False, False]

    cart_comm = comm.Create_cart(
        dims=dims,
        periods=periods,
        reorder=False
    )

    # Get Cartesian coordinates
    coords = cart_comm.Get_coords(rank)

    row_rank = coords[0]
    col_rank = coords[1]

    # Find neighbors along axis 0 (up/down)
    nbr_up, nbr_down = cart_comm.Shift(0, 1)

    # Find neighbors along axis 1 (left/right)
    nbr_left, nbr_right = cart_comm.Shift(1, 1)

    # Select GPU 0
    cuda.select_device(0)

    # Get GPU UUID on rank 0
    gpu_uuid = get_gpu_uuid() if rank == 0 else None

    # Share GPU UUID with all ranks
    gpu_uuid = comm.bcast(gpu_uuid, root=0)

    # Generate student-specific seed
    instance_seed = derive_student_seed(
        github_user,
        gpu_uuid
    )

    # Return distributed context
    return {
        "comm": comm,
        "cart_comm": cart_comm,
        "rank": rank,
        "size": size,
        "coords": coords,
        "row_rank": row_rank,
        "col_rank": col_rank,
        "nbr_up": nbr_up,
        "nbr_down": nbr_down,
        "nbr_left": nbr_left,
        "nbr_right": nbr_right,
        "gpu_uuid": gpu_uuid,
        "instance_seed": instance_seed
    }


# =====================================================
# MAIN EXECUTION
# =====================================================

if __name__ == "__main__":

    GITHUB_USERNAME = "shameke"

    context = init_distributed_context(GITHUB_USERNAME)

    comm = context["comm"]
    rank = context["rank"]

    print("=" * 55)
    print(f"MPI RANK: {rank}")
    print(f"Cartesian coordinates: {context['coords']}")
    print(f"GPU UUID: {context['gpu_uuid']}")
    print(f"INSTANCE_SEED: {context['instance_seed']}")
    print(f"Neighbor UP: {context['nbr_up']}")
    print(f"Neighbor DOWN: {context['nbr_down']}")
    print(f"Neighbor LEFT: {context['nbr_left']}")
    print(f"Neighbor RIGHT: {context['nbr_right']}")
    print("=" * 55)

    comm.Barrier()

    if rank == 0:
        print("TASK 1 COMPLETED SUCCESSFULLY")
        print("MPI processes: 4")
        print("Cartesian grid: 2 x 2")
        print("GPU device: 0")

Overwriting distributed_cuda_solver.py


In [10]:
import subprocess

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "-np", "4",
    "python3",
    "distributed_cuda_solver.py"
]

proc = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print(proc.stdout)

if proc.returncode != 0:
    print("ERROR:")
    print(proc.stderr)
    raise RuntimeError("Task 1 failed")

print("Execution successful!")

MPI RANK: 0
Cartesian coordinates: [0, 0]
GPU UUID: GPU-dd787b66-a73b-5d9a-f7b7-e6321566d0b6
INSTANCE_SEED: 17802426790179078025
Neighbor UP: -2
Neighbor DOWN: 2
Neighbor LEFT: -2
Neighbor RIGHT: 1
MPI RANK: 3
MPI RANK: 1
Cartesian coordinates: [1, 1]
GPU UUID: GPU-dd787b66-a73b-5d9a-f7b7-e6321566d0b6
INSTANCE_SEED: 17802426790179078025
Neighbor UP: 1
Neighbor DOWN: -2
Neighbor LEFT: 2
Neighbor RIGHT: -2
Cartesian coordinates: [0, 1]
GPU UUID: GPU-dd787b66-a73b-5d9a-f7b7-e6321566d0b6
INSTANCE_SEED: 17802426790179078025
Neighbor UP: -2
Neighbor DOWN: 3
Neighbor LEFT: 0
Neighbor RIGHT: -2
MPI RANK: 2
Cartesian coordinates: [1, 0]
GPU UUID: GPU-dd787b66-a73b-5d9a-f7b7-e6321566d0b6
INSTANCE_SEED: 17802426790179078025
Neighbor UP: 0
Neighbor DOWN: -2
Neighbor LEFT: -2
Neighbor RIGHT: 3
TASK 1 COMPLETED SUCCESSFULLY
MPI processes: 4
Cartesian grid: 2 x 2
GPU device: 0

Execution successful!


In [11]:
%%writefile distributed_cuda_solver.py

import hashlib
import subprocess
import numpy as np

from mpi4py import MPI
from numba import cuda

# Constants
GLOBAL_NX = 512
GLOBAL_NY = 512
LOCAL_NX = 256
LOCAL_NY = 256

DX = 1.0 / 513
DY = 1.0 / 513


# =====================================================
# GPU UUID
# =====================================================

def get_gpu_uuid():

    result = subprocess.check_output(
        [
            "nvidia-smi",
            "--query-gpu=uuid",
            "--format=csv,noheader"
        ],
        encoding="utf-8"
    )

    return result.strip().splitlines()[0]


# =====================================================
# STUDENT SEED
# =====================================================

def derive_student_seed(github_user, gpu_uuid):

    digest = hashlib.sha256(
        f"{github_user.strip()}::{gpu_uuid}".encode("utf-8")
    ).hexdigest()

    return int(digest[:16], 16)


# =====================================================
# TASK 1: DISTRIBUTED CONTEXT
# =====================================================

def init_distributed_context(github_user):

    comm = MPI.COMM_WORLD

    rank = comm.Get_rank()
    size = comm.Get_size()

    if size != 4:
        raise RuntimeError(
            f"Expected exactly 4 MPI ranks, got {size}"
        )

    dims = [2, 2]
    periods = [False, False]

    cart_comm = comm.Create_cart(
        dims=dims,
        periods=periods,
        reorder=False
    )

    coords = cart_comm.Get_coords(rank)

    row_rank = coords[0]
    col_rank = coords[1]

    nbr_up, nbr_down = cart_comm.Shift(0, 1)
    nbr_left, nbr_right = cart_comm.Shift(1, 1)

    cuda.select_device(0)

    gpu_uuid = get_gpu_uuid() if rank == 0 else None

    gpu_uuid = comm.bcast(gpu_uuid, root=0)

    instance_seed = derive_student_seed(
        github_user,
        gpu_uuid
    )

    return {
        "comm": comm,
        "cart_comm": cart_comm,
        "rank": rank,
        "size": size,
        "coords": coords,
        "row_rank": row_rank,
        "col_rank": col_rank,
        "nbr_up": nbr_up,
        "nbr_down": nbr_down,
        "nbr_left": nbr_left,
        "nbr_right": nbr_right,
        "gpu_uuid": gpu_uuid,
        "instance_seed": instance_seed
    }


# =====================================================
# TASK 2: SEEDED HETEROGENEOUS SUBDOMAIN INITIALIZATION
# =====================================================

def initialize_subdomains(context):

    row_rank = context["row_rank"]
    col_rank = context["col_rank"]

    seed = context["instance_seed"]

    # Local grid with 1-cell ghost halo
    ny = LOCAL_NY
    nx = LOCAL_NX

    # Global indices including ghost cells
    global_rows = row_rank * ny + np.arange(258)
    global_cols = col_rank * nx + np.arange(258)

    # Physical coordinates
    y = global_rows * DY
    x = global_cols * DX

    X, Y = np.meshgrid(x, y)

    # -----------------------------------------------
    # Spatially heterogeneous diffusivity
    # Range: [0.08, 0.24]
    # -----------------------------------------------

    variation = (
        0.5
        + 0.25 * np.sin(2 * np.pi * X)
        * np.cos(2 * np.pi * Y)
    )

    variation = np.clip(variation, 0.0, 1.0)

    kappa = 0.08 + 0.16 * variation

    kappa = kappa.astype(np.float64)

    # -----------------------------------------------
    # Seeded Gaussian thermal pulses
    # -----------------------------------------------

    rng = np.random.default_rng(seed)

    num_pulses = 4

    centers_x = rng.uniform(0.15, 0.85, num_pulses)
    centers_y = rng.uniform(0.15, 0.85, num_pulses)

    amplitudes = rng.uniform(0.5, 1.0, num_pulses)

    sigmas = rng.uniform(0.035, 0.075, num_pulses)

    u = np.zeros((258, 258), dtype=np.float64)

    for i in range(num_pulses):

        dx2 = (X - centers_x[i]) ** 2
        dy2 = (Y - centers_y[i]) ** 2

        gaussian = np.exp(
            -(dx2 + dy2) / (2 * sigmas[i] ** 2)
        )

        u += amplitudes[i] * gaussian

    # -----------------------------------------------
    # Localized source field S(x,y)
    # -----------------------------------------------

    source = np.zeros((258, 258), dtype=np.float64)

    source_x = 0.5
    source_y = 0.5
    source_sigma = 0.08

    source = 0.2 * np.exp(
        -(
            (X - source_x) ** 2
            + (Y - source_y) ** 2
        ) / (2 * source_sigma ** 2)
    )

    # -----------------------------------------------
    # Enforce zero outer physical boundaries
    # -----------------------------------------------

    if context["nbr_up"] == MPI.PROC_NULL:
        u[0, :] = 0.0

    if context["nbr_down"] == MPI.PROC_NULL:
        u[ny + 1, :] = 0.0

    if context["nbr_left"] == MPI.PROC_NULL:
        u[:, 0] = 0.0

    if context["nbr_right"] == MPI.PROC_NULL:
        u[:, nx + 1] = 0.0

    return u, kappa, source, X, Y


# =====================================================
# MAIN EXECUTION
# =====================================================

if __name__ == "__main__":

    GITHUB_USERNAME = "shameke"

    # Task 1
    context = init_distributed_context(GITHUB_USERNAME)

    rank = context["rank"]
    comm = context["comm"]

    print("=" * 55)
    print(f"MPI RANK: {rank}")
    print(f"Coordinates: {context['coords']}")
    print(f"GPU UUID: {context['gpu_uuid']}")
    print(f"INSTANCE_SEED: {context['instance_seed']}")
    print(f"UP: {context['nbr_up']}")
    print(f"DOWN: {context['nbr_down']}")
    print(f"LEFT: {context['nbr_left']}")
    print(f"RIGHT: {context['nbr_right']}")

    # Task 2
    u, kappa, source, X, Y = initialize_subdomains(context)

    print("-" * 55)
    print(f"Rank {rank}: TASK 2 RESULTS")
    print("Temperature shape:", u.shape)
    print("Kappa shape:", kappa.shape)
    print("Source shape:", source.shape)

    print("Temperature min:", np.min(u))
    print("Temperature max:", np.max(u))

    print("Kappa min:", np.min(kappa))
    print("Kappa max:", np.max(kappa))

    print("Source max:", np.max(source))

    print("=" * 55)

    comm.Barrier()

    if rank == 0:
        print("TASK 1 COMPLETED SUCCESSFULLY")
        print("TASK 2 COMPLETED SUCCESSFULLY")

Overwriting distributed_cuda_solver.py


In [12]:
import subprocess

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "-np", "4",
    "python3",
    "distributed_cuda_solver.py"
]

proc = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print(proc.stdout)

if proc.returncode != 0:
    print("ERROR:")
    print(proc.stderr)
    raise RuntimeError("Task 1 or Task 2 failed")

print("Execution successful!")

MPI RANK: 0
Coordinates: [0, 0]
GPU UUID: GPU-dd787b66-a73b-5d9a-f7b7-e6321566d0b6
INSTANCE_SEED: 17802426790179078025
UP: -2
DOWN: 2
LEFT: -2
RIGHT: 1
MPI RANK: 2
Coordinates: [1, 0]
GPU UUID: GPU-dd787b66-a73b-5d9a-f7b7-e6321566d0b6
INSTANCE_SEED: 17802426790179078025
UP: 0
DOWN: -2
LEFT: -2
RIGHT: 3
MPI RANK: 1
Coordinates: [0, 1]
GPU UUID: GPU-dd787b66-a73b-5d9a-f7b7-e6321566d0b6
INSTANCE_SEED: 17802426790179078025
UP: -2
DOWN: 3
LEFT: 0
RIGHT: -2
MPI RANK: 3
Coordinates: [1, 1]
GPU UUID: GPU-dd787b66-a73b-5d9a-f7b7-e6321566d0b6
INSTANCE_SEED: 17802426790179078025
UP: 1
DOWN: -2
LEFT: 2
RIGHT: -2
-------------------------------------------------------
-------------------------------------------------------
Rank 0: TASK 2 RESULTS
Temperature shape: (258, 258)
Kappa shape: (258, 258)
Source shape: (258, 258)
Temperature min: 0.0
Temperature max: 0.6029541602203299
Kappa min: 0.12000093756684575
Kappa max: 0.199999812485576
Source max: 0.1999703159558338
Rank 1: TASK 2 RESULTS
Tempera

In [14]:
%%writefile distributed_cuda_solver.py

import hashlib
import subprocess
import numpy as np

from mpi4py import MPI
from numba import cuda

# =====================================================
# CONSTANTS
# =====================================================

GLOBAL_NX = 512
GLOBAL_NY = 512

LOCAL_NX = 256
LOCAL_NY = 256

DX = 1.0 / 513
DY = 1.0 / 513

THREADS = 256

TAG_UP = 10
TAG_DOWN = 11
TAG_LEFT = 12
TAG_RIGHT = 13


# =====================================================
# GPU UUID AND SEED
# =====================================================

def get_gpu_uuid():

    result = subprocess.check_output(
        [
            "nvidia-smi",
            "--query-gpu=uuid",
            "--format=csv,noheader"
        ],
        encoding="utf-8"
    )

    return result.strip().splitlines()[0]


def derive_student_seed(github_user, gpu_uuid):

    digest = hashlib.sha256(
        f"{github_user.strip()}::{gpu_uuid}".encode("utf-8")
    ).hexdigest()

    return int(digest[:16], 16)


# =====================================================
# TASK 1: DISTRIBUTED CONTEXT
# =====================================================

def init_distributed_context(github_user):

    comm = MPI.COMM_WORLD

    rank = comm.Get_rank()
    size = comm.Get_size()

    if size != 4:
        raise RuntimeError(
            f"Expected exactly 4 MPI ranks, got {size}"
        )

    cart_comm = comm.Create_cart(
        dims=[2, 2],
        periods=[False, False],
        reorder=False
    )

    coords = cart_comm.Get_coords(rank)

    row_rank, col_rank = coords

    nbr_up, nbr_down = cart_comm.Shift(0, 1)
    nbr_left, nbr_right = cart_comm.Shift(1, 1)

    cuda.select_device(0)

    gpu_uuid = get_gpu_uuid() if rank == 0 else None
    gpu_uuid = comm.bcast(gpu_uuid, root=0)

    instance_seed = derive_student_seed(
        github_user,
        gpu_uuid
    )

    return {
        "comm": comm,
        "cart_comm": cart_comm,
        "rank": rank,
        "size": size,
        "coords": coords,
        "row_rank": row_rank,
        "col_rank": col_rank,
        "nbr_up": nbr_up,
        "nbr_down": nbr_down,
        "nbr_left": nbr_left,
        "nbr_right": nbr_right,
        "gpu_uuid": gpu_uuid,
        "instance_seed": instance_seed
    }


# =====================================================
# TASK 2: SUBDOMAIN INITIALIZATION
# =====================================================

def initialize_subdomains(context):

    row_rank = context["row_rank"]
    col_rank = context["col_rank"]

    seed = context["instance_seed"]

    ny = LOCAL_NY
    nx = LOCAL_NX

    global_rows = row_rank * ny + np.arange(258)
    global_cols = col_rank * nx + np.arange(258)

    y = global_rows * DY
    x = global_cols * DX

    X, Y = np.meshgrid(x, y)

    variation = (
        0.5
        + 0.25 * np.sin(2 * np.pi * X)
        * np.cos(2 * np.pi * Y)
    )

    variation = np.clip(variation, 0.0, 1.0)

    kappa = (
        0.08 + 0.16 * variation
    ).astype(np.float64)

    rng = np.random.default_rng(seed)

    num_pulses = 4

    centers_x = rng.uniform(0.15, 0.85, num_pulses)
    centers_y = rng.uniform(0.15, 0.85, num_pulses)

    amplitudes = rng.uniform(0.5, 1.0, num_pulses)
    sigmas = rng.uniform(0.035, 0.075, num_pulses)

    u = np.zeros((258, 258), dtype=np.float64)

    for i in range(num_pulses):

        gaussian = np.exp(
            -(
                (X - centers_x[i]) ** 2
                + (Y - centers_y[i]) ** 2
            ) / (2 * sigmas[i] ** 2)
        )

        u += amplitudes[i] * gaussian

    source = 0.2 * np.exp(
        -(
            (X - 0.5) ** 2
            + (Y - 0.5) ** 2
        ) / (2 * 0.08 ** 2)
    )

    # Physical boundaries
    if context["nbr_up"] == MPI.PROC_NULL:
        u[0, :] = 0.0

    if context["nbr_down"] == MPI.PROC_NULL:
        u[ny + 1, :] = 0.0

    if context["nbr_left"] == MPI.PROC_NULL:
        u[:, 0] = 0.0

    if context["nbr_right"] == MPI.PROC_NULL:
        u[:, nx + 1] = 0.0

    return u, kappa, source, X, Y


# =====================================================
# TASK 3: GPU HALO PACK KERNEL
# =====================================================

@cuda.jit
def pack_halos_kernel(
    u,
    send_up,
    send_down,
    send_left,
    send_right
):

    idx = cuda.grid(1)

    if idx < LOCAL_NX:

        # Top row
        send_up[idx] = u[1, idx + 1]

        # Bottom row
        send_down[idx] = u[LOCAL_NY, idx + 1]

    if idx < LOCAL_NY:

        # Left column
        send_left[idx] = u[idx + 1, 1]

        # Right column
        send_right[idx] = u[idx + 1, LOCAL_NX]


# =====================================================
# TASK 3: GPU HALO UNPACK KERNEL
# =====================================================

@cuda.jit
def unpack_halos_kernel(
    u,
    recv_buffer,
    direction
):

    idx = cuda.grid(1)

    if direction == 0:

        # Received from upper neighbor
        if idx < LOCAL_NX:
            u[0, idx + 1] = recv_buffer[idx]

    elif direction == 1:

        # Received from lower neighbor
        if idx < LOCAL_NX:
            u[LOCAL_NY + 1, idx + 1] = recv_buffer[idx]

    elif direction == 2:

        # Received from left neighbor
        if idx < LOCAL_NY:
            u[idx + 1, 0] = recv_buffer[idx]

    elif direction == 3:

        # Received from right neighbor
        if idx < LOCAL_NY:
            u[idx + 1, LOCAL_NX + 1] = recv_buffer[idx]


# =====================================================
# TASK 3: ASYNCHRONOUS HALO EXCHANGE
# =====================================================

def exchange_halos_async(context, d_u):

    cart_comm = context["cart_comm"]

    # Neighbor ranks
    up = context["nbr_up"]
    down = context["nbr_down"]
    left = context["nbr_left"]
    right = context["nbr_right"]

    # CUDA stream
    stream_boundary = cuda.stream()

    # Device send buffers
    d_send_up = cuda.device_array(LOCAL_NX, dtype=np.float64)
    d_send_down = cuda.device_array(LOCAL_NX, dtype=np.float64)
    d_send_left = cuda.device_array(LOCAL_NY, dtype=np.float64)
    d_send_right = cuda.device_array(LOCAL_NY, dtype=np.float64)

    # Device receive buffers
    d_recv_up = cuda.device_array(LOCAL_NX, dtype=np.float64)
    d_recv_down = cuda.device_array(LOCAL_NX, dtype=np.float64)
    d_recv_left = cuda.device_array(LOCAL_NY, dtype=np.float64)
    d_recv_right = cuda.device_array(LOCAL_NY, dtype=np.float64)

    # Pinned host send buffers
    h_send_up = cuda.pinned_array(LOCAL_NX, dtype=np.float64)
    h_send_down = cuda.pinned_array(LOCAL_NX, dtype=np.float64)
    h_send_left = cuda.pinned_array(LOCAL_NY, dtype=np.float64)
    h_send_right = cuda.pinned_array(LOCAL_NY, dtype=np.float64)

    # Pinned host receive buffers
    h_recv_up = cuda.pinned_array(LOCAL_NX, dtype=np.float64)
    h_recv_down = cuda.pinned_array(LOCAL_NX, dtype=np.float64)
    h_recv_left = cuda.pinned_array(LOCAL_NY, dtype=np.float64)
    h_recv_right = cuda.pinned_array(LOCAL_NY, dtype=np.float64)

    threads = THREADS
    blocks = (max(LOCAL_NX, LOCAL_NY) + threads - 1) // threads

    # -----------------------------------------------
    # Stage A: Pack borders on GPU
    # -----------------------------------------------

    pack_halos_kernel[blocks, threads, stream_boundary](
        d_u,
        d_send_up,
        d_send_down,
        d_send_left,
        d_send_right
    )

    # Async Device-to-Host transfers
    d_send_up.copy_to_host(h_send_up, stream=stream_boundary)
    d_send_down.copy_to_host(h_send_down, stream=stream_boundary)
    d_send_left.copy_to_host(h_send_left, stream=stream_boundary)
    d_send_right.copy_to_host(h_send_right, stream=stream_boundary)

    # Ensure host send buffers are ready
    stream_boundary.synchronize()

    # -----------------------------------------------
    # Stage B: Non-blocking MPI communication
    # -----------------------------------------------

    requests = []

    # Receive from all four neighbors
    requests.append(
        cart_comm.Irecv(h_recv_up, source=up, tag=TAG_DOWN)
    )

    requests.append(
        cart_comm.Irecv(h_recv_down, source=down, tag=TAG_UP)
    )

    requests.append(
        cart_comm.Irecv(h_recv_left, source=left, tag=TAG_RIGHT)
    )

    requests.append(
        cart_comm.Irecv(h_recv_right, source=right, tag=TAG_LEFT)
    )

    # Send to all four neighbors
    requests.append(
        cart_comm.Isend(h_send_up, dest=up, tag=TAG_UP)
    )

    requests.append(
        cart_comm.Isend(h_send_down, dest=down, tag=TAG_DOWN)
    )

    requests.append(
        cart_comm.Isend(h_send_left, dest=left, tag=TAG_LEFT)
    )

    requests.append(
        cart_comm.Isend(h_send_right, dest=right, tag=TAG_RIGHT)
    )

    # Complete all MPI operations
    MPI.Request.Waitall(requests)

    # -----------------------------------------------
    # Stage C: Host-to-Device transfers
    # -----------------------------------------------

    # Only copy data from real neighbors
    if up != MPI.PROC_NULL:
        d_recv_up.copy_to_device(h_recv_up, stream=stream_boundary)

    if down != MPI.PROC_NULL:
        d_recv_down.copy_to_device(h_recv_down, stream=stream_boundary)

    if left != MPI.PROC_NULL:
        d_recv_left.copy_to_device(h_recv_left, stream=stream_boundary)

    if right != MPI.PROC_NULL:
        d_recv_right.copy_to_device(h_recv_right, stream=stream_boundary)

    # -----------------------------------------------
    # Unpack received halos
    # -----------------------------------------------

    if up != MPI.PROC_NULL:
        unpack_halos_kernel[blocks, threads, stream_boundary](
            d_u, d_recv_up, 0
        )

    if down != MPI.PROC_NULL:
        unpack_halos_kernel[blocks, threads, stream_boundary](
            d_u, d_recv_down, 1
        )

    if left != MPI.PROC_NULL:
        unpack_halos_kernel[blocks, threads, stream_boundary](
            d_u, d_recv_left, 2
        )

    if right != MPI.PROC_NULL:
        unpack_halos_kernel[blocks, threads, stream_boundary](
            d_u, d_recv_right, 3
        )

    stream_boundary.synchronize()

    return stream_boundary


# =====================================================
# MAIN EXECUTION
# =====================================================

if __name__ == "__main__":

    GITHUB_USERNAME = "shameke"

    # Task 1
    context = init_distributed_context(GITHUB_USERNAME)

    rank = context["rank"]
    comm = context["comm"]

    # Task 2
    u, kappa, source, X, Y = initialize_subdomains(context)

    # Copy local temperature field to GPU
    d_u = cuda.to_device(u)

    # Task 3
    exchange_halos_async(context, d_u)

    # Copy updated array back for verification
    result = d_u.copy_to_host()

    print("=" * 55)
    print(f"RANK: {rank}")
    print("Local grid:", u.shape)
    print("GPU UUID:", context["gpu_uuid"])
    print("INSTANCE_SEED:", context["instance_seed"])
    print("Temperature min:", np.min(result))
    print("Temperature max:", np.max(result))
    print("Kappa min:", np.min(kappa))
    print("Kappa max:", np.max(kappa))
    print("Halo exchange completed")
    print("=" * 55)

    comm.Barrier()

    if rank == 0:
        print("TASK 1 COMPLETED SUCCESSFULLY")
        print("TASK 2 COMPLETED SUCCESSFULLY")
        print("TASK 3 COMPLETED SUCCESSFULLY")

Overwriting distributed_cuda_solver.py


In [15]:
import subprocess

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "-np", "4",
    "python3",
    "distributed_cuda_solver.py"
]

proc = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print(proc.stdout)

if proc.returncode != 0:
    print("ERROR:")
    print(proc.stderr)
    raise RuntimeError("Task 3 failed")

print("Execution successful!")

RANK: 2
RANK: 1
Local grid: (258, 258)
GPU UUID: GPU-dd787b66-a73b-5d9a-f7b7-e6321566d0b6
INSTANCE_SEED: 17802426790179078025
RANK: 3
Local grid: (258, 258)
GPU UUID: GPU-dd787b66-a73b-5d9a-f7b7-e6321566d0b6
INSTANCE_SEED: 17802426790179078025
Temperature min: 0.0
Temperature min: 0.0
Local grid: (258, 258)
GPU UUID: GPU-dd787b66-a73b-5d9a-f7b7-e6321566d0b6
INSTANCE_SEED: 17802426790179078025
Temperature min: 0.0
Temperature max: 0.7662153047115398
Temperature max: 0.7662153047115398
Kappa min: 0.120000187514424
Kappa max: 0.19999906243315427
Halo exchange completed
Temperature max: 0.09267964853740712
Kappa min: 0.120000187514424
Kappa max: 0.19999906243315427
Halo exchange completed
RANK: 0
Local grid: (258, 258)
GPU UUID: GPU-dd787b66-a73b-5d9a-f7b7-e6321566d0b6
INSTANCE_SEED: 17802426790179078025
Temperature min: 0.0
Temperature max: 0.6029541602203299
Kappa min: 0.12000093756684575
Kappa min: 0.12000093756684575
Kappa max: 0.199999812485576
Kappa max: 0.199999812485576
Halo exchan

In [16]:
%%writefile distributed_cuda_solver.py
import sys
import hashlib
import subprocess
import numpy as np

from mpi4py import MPI
from numba import cuda, float64


# ============================================================
# CONFIGURATION
# ============================================================

GLOBAL_NX = 512
GLOBAL_NY = 512

LOCAL_NX = 256
LOCAL_NY = 256

TILE_DIM = 16
SHARED_DIM = 18

LAMBDA_DAMP = 0.15
DT = 1.0e-7

TAG_UP = 10
TAG_DOWN = 11
TAG_LEFT = 12
TAG_RIGHT = 13


# ============================================================
# TASK 1: GPU UUID AND STUDENT SEED
# ============================================================

def get_gpu_uuid():
    result = subprocess.check_output(
        [
            "nvidia-smi",
            "--query-gpu=uuid",
            "--format=csv,noheader"
        ],
        encoding="utf-8"
    )
    return result.strip().splitlines()[0].strip()


def derive_student_seed(github_user, gpu_uuid):
    value = f"{github_user.strip()}::{gpu_uuid}"
    digest = hashlib.sha256(value.encode("utf-8")).hexdigest()
    return int(digest[:16], 16)


def init_distributed_context(github_user):

    comm = MPI.COMM_WORLD
    rank = comm.Get_rank()
    size = comm.Get_size()

    if size != 4:
        raise RuntimeError(
            f"Expected 4 MPI processes, got {size}"
        )

    cart = comm.Create_cart(
        dims=[2, 2],
        periods=[False, False],
        reorder=False
    )

    coords = cart.Get_coords(rank)

    nbr_up, nbr_down = cart.Shift(0, 1)
    nbr_left, nbr_right = cart.Shift(1, 1)

    cuda.select_device(0)

    gpu_uuid = get_gpu_uuid() if rank == 0 else None
    gpu_uuid = comm.bcast(gpu_uuid, root=0)

    instance_seed = derive_student_seed(
        github_user,
        gpu_uuid
    )

    return {
        "comm": cart,
        "rank": rank,
        "coords": coords,
        "nbr_up": nbr_up,
        "nbr_down": nbr_down,
        "nbr_left": nbr_left,
        "nbr_right": nbr_right,
        "gpu_uuid": gpu_uuid,
        "instance_seed": instance_seed
    }


# ============================================================
# TASK 2: SUBDOMAIN INITIALIZATION
# ============================================================

def initialize_subdomains(context):

    row_rank, col_rank = context["coords"]

    global_rows = (
        row_rank * LOCAL_NY
        + np.arange(258)
    )

    global_cols = (
        col_rank * LOCAL_NX
        + np.arange(258)
    )

    y = global_rows / 513.0
    x = global_cols / 513.0

    X, Y = np.meshgrid(x, y)

    # Heterogeneous diffusivity
    kappa = (
        0.08
        + 0.16 * np.clip(
            0.5
            + 0.25
            * np.sin(2.0 * np.pi * X)
            * np.cos(2.0 * np.pi * Y),
            0.0,
            1.0
        )
    )

    rng = np.random.default_rng(
        context["instance_seed"]
    )

    u = np.zeros((258, 258), dtype=np.float64)

    # Four Gaussian thermal pulses
    for _ in range(4):

        cx = rng.uniform(0.15, 0.85)
        cy = rng.uniform(0.15, 0.85)

        amplitude = rng.uniform(0.5, 1.0)
        sigma = rng.uniform(0.035, 0.075)

        u += amplitude * np.exp(
            -(
                (X - cx) ** 2
                + (Y - cy) ** 2
            ) / (2.0 * sigma ** 2)
        )

    # Localized source
    source = 0.2 * np.exp(
        -(
            (X - 0.5) ** 2
            + (Y - 0.5) ** 2
        ) / (2.0 * 0.08 ** 2)
    )

    # Physical Dirichlet boundaries
    if context["nbr_up"] == MPI.PROC_NULL:
        u[0, :] = 0.0

    if context["nbr_down"] == MPI.PROC_NULL:
        u[257, :] = 0.0

    if context["nbr_left"] == MPI.PROC_NULL:
        u[:, 0] = 0.0

    if context["nbr_right"] == MPI.PROC_NULL:
        u[:, 257] = 0.0

    return u, kappa, source


# ============================================================
# TASK 3: CUDA HALO PACKING
# ============================================================

@cuda.jit
def pack_halos_kernel(
    u,
    send_up,
    send_down,
    send_left,
    send_right
):

    idx = cuda.grid(1)

    if idx < LOCAL_NX:
        send_up[idx] = u[1, idx + 1]
        send_down[idx] = u[256, idx + 1]

    if idx < LOCAL_NY:
        send_left[idx] = u[idx + 1, 1]
        send_right[idx] = u[idx + 1, 256]


@cuda.jit
def unpack_halos_kernel(u, recv_buffer, direction):

    idx = cuda.grid(1)

    if idx < 256:

        if direction == 0:
            u[0, idx + 1] = recv_buffer[idx]

        elif direction == 1:
            u[257, idx + 1] = recv_buffer[idx]

        elif direction == 2:
            u[idx + 1, 0] = recv_buffer[idx]

        elif direction == 3:
            u[idx + 1, 257] = recv_buffer[idx]


def start_halo_exchange(context, d_u, stream_boundary):

    rank = context["rank"]
    cart = context["comm"]

    # Device send buffers
    d_send = [
        cuda.device_array(256, dtype=np.float64)
        for _ in range(4)
    ]

    # Device receive buffers
    d_recv = [
        cuda.device_array(256, dtype=np.float64)
        for _ in range(4)
    ]

    # Pinned host buffers
    h_send = [
        cuda.pinned_array(256, dtype=np.float64)
        for _ in range(4)
    ]

    h_recv = [
        cuda.pinned_array(256, dtype=np.float64)
        for _ in range(4)
    ]

    blocks = (256 + 255) // 256

    pack_halos_kernel[blocks, 256, stream_boundary](
        d_u,
        d_send[0],
        d_send[1],
        d_send[2],
        d_send[3]
    )

    # Asynchronous device-to-host transfers
    for i in range(4):
        d_send[i].copy_to_host(
            h_send[i],
            stream=stream_boundary
        )

    stream_boundary.synchronize()

    neighbors = [
        context["nbr_up"],
        context["nbr_down"],
        context["nbr_left"],
        context["nbr_right"]
    ]

    # Receive tags are opposite to the sender's direction
    recv_tags = [
        TAG_DOWN,
        TAG_UP,
        TAG_RIGHT,
        TAG_LEFT
    ]

    send_tags = [
        TAG_UP,
        TAG_DOWN,
        TAG_LEFT,
        TAG_RIGHT
    ]

    requests = []

    for i in range(4):
        requests.append(
            cart.Irecv(
                [h_recv[i], MPI.DOUBLE],
                source=neighbors[i],
                tag=recv_tags[i]
            )
        )

    for i in range(4):
        requests.append(
            cart.Isend(
                [h_send[i], MPI.DOUBLE],
                dest=neighbors[i],
                tag=send_tags[i]
            )
        )

    return {
        "requests": requests,
        "d_recv": d_recv,
        "h_recv": h_recv,
        "neighbors": neighbors,
        "rank": rank
    }


def finish_halo_exchange(exchange, d_u, stream_boundary):

    MPI.Request.Waitall(exchange["requests"])

    d_recv = exchange["d_recv"]
    h_recv = exchange["h_recv"]
    neighbors = exchange["neighbors"]

    for i in range(4):

        if neighbors[i] != MPI.PROC_NULL:

            d_recv[i].copy_to_device(
                h_recv[i],
                stream=stream_boundary
            )

            unpack_halos_kernel[1, 256, stream_boundary](
                d_u,
                d_recv[i],
                i
            )

    stream_boundary.synchronize()


# ============================================================
# TASK 4: SHARED-MEMORY TILED STENCIL
# ============================================================

@cuda.jit
def tiled_stencil_kernel(
    u,
    kappa,
    source,
    output,
    r_start,
    r_end,
    c_start,
    c_end,
    boundary_mode
):

    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y

    row = r_start + cuda.blockIdx.y * 16 + ty
    col = c_start + cuda.blockIdx.x * 16 + tx

    sm_u = cuda.shared.array(
        shape=(18, 18),
        dtype=float64
    )

    sm_k = cuda.shared.array(
        shape=(18, 18),
        dtype=float64
    )

    # Center cell
    rr = row
    cc = col

    if rr < 258 and cc < 258:
        sm_u[ty + 1, tx + 1] = u[rr, cc]
        sm_k[ty + 1, tx + 1] = kappa[rr, cc]
    else:
        sm_u[ty + 1, tx + 1] = 0.0
        sm_k[ty + 1, tx + 1] = 0.08

    # Top apron
    if ty == 0:
        if row > 0 and col < 258:
            sm_u[0, tx + 1] = u[row - 1, col]
            sm_k[0, tx + 1] = kappa[row - 1, col]
        else:
            sm_u[0, tx + 1] = 0.0
            sm_k[0, tx + 1] = 0.08

    # Bottom apron
    if ty == 15:
        if row + 1 < 258 and col < 258:
            sm_u[17, tx + 1] = u[row + 1, col]
            sm_k[17, tx + 1] = kappa[row + 1, col]
        else:
            sm_u[17, tx + 1] = 0.0
            sm_k[17, tx + 1] = 0.08

    # Left apron
    if tx == 0:
        if col > 0 and row < 258:
            sm_u[ty + 1, 0] = u[row, col - 1]
            sm_k[ty + 1, 0] = kappa[row, col - 1]
        else:
            sm_u[ty + 1, 0] = 0.0
            sm_k[ty + 1, 0] = 0.08

    # Right apron
    if tx == 15:
        if col + 1 < 258 and row < 258:
            sm_u[ty + 1, 17] = u[row, col + 1]
            sm_k[ty + 1, 17] = kappa[row, col + 1]
        else:
            sm_u[ty + 1, 17] = 0.0
            sm_k[ty + 1, 17] = 0.08

    cuda.syncthreads()

    # Only owned cells
    if not (
        r_start <= row < r_end
        and c_start <= col < c_end
    ):
        return

    if row < 1 or row > 256 or col < 1 or col > 256:
        return

    # Deep interior or perimeter selection
    is_boundary = (
        row == 1 or row == 256
        or col == 1 or col == 256
    )

    if boundary_mode == 0 and is_boundary:
        return

    if boundary_mode == 1 and not is_boundary:
        return

    center = sm_u[ty + 1, tx + 1]

    up = sm_u[ty, tx + 1]
    down = sm_u[ty + 2, tx + 1]
    left = sm_u[ty + 1, tx]
    right = sm_u[ty + 1, tx + 2]

    k_center = sm_k[ty + 1, tx + 1]

    k_up = 0.5 * (
        k_center + sm_k[ty, tx + 1]
    )

    k_down = 0.5 * (
        k_center + sm_k[ty + 2, tx + 1]
    )

    k_left = 0.5 * (
        k_center + sm_k[ty + 1, tx]
    )

    k_right = 0.5 * (
        k_center + sm_k[ty + 1, tx + 2]
    )

    dx = 1.0 / 513.0
    dy = 1.0 / 513.0

    flux_x = (
        k_down * (down - center)
        - k_up * (center - up)
    ) / (dx * dx)

    flux_y = (
        k_right * (right - center)
        - k_left * (center - left)
    ) / (dy * dy)

    reaction = -0.15 * center * center * center

    result = center + DT * (
        flux_x
        + flux_y
        + reaction
        + source[row, col]
    )

    output[row, col] = result


# ============================================================
# TASK 4: DUAL-STREAM OVERLAP
# ============================================================

def step_overlapped(
    context,
    d_u,
    d_kappa,
    d_source,
    d_output
):

    stream_boundary = cuda.stream()
    stream_interior = cuda.stream()

    # Stage A: pack halos and start MPI communication
    exchange = start_halo_exchange(
        context,
        d_u,
        stream_boundary
    )

    # Stage B: compute deep interior while MPI is working
    tiled_stencil_kernel[
        (16, 16),
        (16, 16),
        stream_interior
    ](
        d_u,
        d_kappa,
        d_source,
        d_output,
        2, 256,
        2, 256,
        0
    )

    # Stage C: wait for communication and unpack halos
    finish_halo_exchange(
        exchange,
        d_u,
        stream_boundary
    )

    # Compute perimeter ring
    tiled_stencil_kernel[
        (16, 16),
        (16, 16),
        stream_boundary
    ](
        d_u,
        d_kappa,
        d_source,
        d_output,
        1, 257,
        1, 257,
        1
    )

    # Ensure both streams have completed before swapping arrays
    stream_interior.synchronize()
    stream_boundary.synchronize()


# ============================================================
# MAIN: TASKS 1–4 TEST
# ============================================================

def main():

    github_user = (
        sys.argv[1]
        if len(sys.argv) > 1
        else "shameke"
    )

    context = init_distributed_context(github_user)

    u, kappa, source = initialize_subdomains(context)

    d_u = cuda.to_device(u)
    d_kappa = cuda.to_device(kappa)
    d_source = cuda.to_device(source)

    d_output = cuda.to_device(
        np.zeros((258, 258), dtype=np.float64)
    )

    # Test one overlapped stencil step
    step_overlapped(
        context,
        d_u,
        d_kappa,
        d_source,
        d_output
    )

    result = d_output.copy_to_host()

    # Report
    print("=" * 55)
    print("RANK:", context["rank"])
    print("Cartesian coordinates:", context["coords"])
    print("Local grid:", u.shape)
    print("GPU UUID:", context["gpu_uuid"])
    print("INSTANCE_SEED:", context["instance_seed"])
    print("Temperature min:", float(np.min(u)))
    print("Temperature max:", float(np.max(u)))
    print("Kappa min:", float(np.min(kappa)))
    print("Kappa max:", float(np.max(kappa)))
    print("Stencil output min:", float(np.min(result)))
    print("Stencil output max:", float(np.max(result)))
    print("Overlapped step completed")
    print("=" * 55)

    context["comm"].Barrier()

    if context["rank"] == 0:
        print("TASK 1 COMPLETED SUCCESSFULLY")
        print("TASK 2 COMPLETED SUCCESSFULLY")
        print("TASK 3 COMPLETED SUCCESSFULLY")
        print("TASK 4 COMPLETED SUCCESSFULLY")


if __name__ == "__main__":
    main()

Overwriting distributed_cuda_solver.py


In [17]:
import subprocess

GITHUB_USERNAME = "shameke"

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "-np", "4",
    "python3",
    "distributed_cuda_solver.py",
    GITHUB_USERNAME
]

proc = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print(proc.stdout)

if proc.returncode != 0:
    print("STDERR:")
    print(proc.stderr)
    raise RuntimeError("Task 4 failed")

print("Execution successful!")

RANK: 2
Cartesian coordinates: [1, 0]
Local grid: (258, 258)
GPU UUID: GPU-dd787b66-a73b-5d9a-f7b7-e6321566d0b6
INSTANCE_SEED: 17802426790179078025
Temperature min: 0.0
Temperature max: 0.6722178324547461
Kappa min: 0.12000093756684575
Kappa max: 0.199999812485576
Stencil output min: 0.0
Stencil output max: 0.6722019119143615
Overlapped step completed
RANK: 1
Cartesian coordinates: [0, 1]
Local grid: (258, 258)
GPU UUID: GPU-dd787b66-a73b-5d9a-f7b7-e6321566d0b6
INSTANCE_SEED: 17802426790179078025
Temperature min: 0.0
Temperature max: 0.8412794528597525
Kappa min: 0.120000187514424
Kappa max: 0.19999906243315427
Stencil output min: 0.0
Stencil output max: 0.8348810199517044
Overlapped step completed
RANK: 0
Cartesian coordinates: [0, 0]
Local grid: (258, 258)
GPU UUID: GPU-dd787b66-a73b-5d9a-f7b7-e6321566d0b6
INSTANCE_SEED: 17802426790179078025
Temperature min: 0.0
Temperature max: 0.852617065597862
Kappa min: 0.12000093756684575
Kappa max: 0.199999812485576
Stencil output min: 0.0
Sten

In [20]:
from google.colab import files

uploaded = files.upload()

Saving distributed_cuda_solver_task5.py to distributed_cuda_solver_task5.py


In [21]:
import os

os.rename(
    "distributed_cuda_solver_task5.py",
    "distributed_cuda_solver.py"
)

print("File ready:", os.path.exists("distributed_cuda_solver.py"))

File ready: True


In [22]:
!pip install numpy numba mpi4py

In [23]:
!apt-get update -qq
!apt-get install -y -qq openmpi-bin libopenmpi-dev

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [26]:
import subprocess
import os

print("Checking files...")
print("Python file exists:", os.path.exists("distributed_cuda_solver.py"))

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "-np", "4",
    "python3",
    "-u",
    "distributed_cuda_solver.py",
    "shameke"
]

print("\nRunning MPI...")
print("Command:", " ".join(cmd))

proc = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print("\n========== RETURN CODE ==========")
print(proc.returncode)

print("\n========== STDOUT ==========")
print(repr(proc.stdout))

print("\n========== STDERR ==========")
print(repr(proc.stderr))

Checking files...
Python file exists: True

Running MPI...
Command: mpirun --allow-run-as-root --oversubscribe -np 4 python3 -u distributed_cuda_solver.py shameke

========== RETURN CODE ==========
1

========== STDOUT ==========
''

========== STDERR ==========
''
